[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_29_mlp_pure_solution.ipynb)

# 🟢 Solution: SwiGLU MLP without Flax

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `b_29_mlp_pure.ipynb` first.

---
Problem 15's SwiGLU feed-forward block with no Flax.

### Signature
```python
class SwiGLUMLP:
    def __init__(self, d_model, d_ff, *, key): ...
    def __call__(self, x): ...        # (..., d_model) -> (..., d_model)
```

Three projections, named as in problem 15:

| | shape |
|---|---|
| `self.gate_proj` | `(d_model, d_ff)` |
| `self.up_proj` | `(d_model, d_ff)` |
| `self.down_proj` | `(d_ff, d_model)` |

Built from `jax.random.split(key, 3)` — one key reused three times gives three
identical matrices, which makes `gate` and `up` the same tensor and quietly
turns SwiGLU into `silu(z) * z`.

### The formula
$$\text{SwiGLU}(x) = W_{\text{down}}\big(\text{silu}(W_{\text{gate}}x)\odot W_{\text{up}}x\big)$$

One line. The `*` is **elementwise**, which is exactly why `gate_proj` and
`up_proj` must produce the same width — the gate multiplies the value stream
channel by channel.

### Why two projections up and one down
A vanilla MLP is `down(gelu(up(x)))` — two matrices. SwiGLU spends three, so
implementations shrink `d_ff` (Llama uses about `8/3 · d_model` instead of
`4 · d_model`) to keep the parameter count level. `silu(x) = x · sigmoid(x)`,
which you built in `b_22`.

### Why this exists alongside problem 15
Interview sandboxes ship `jax` but not `flax`. Same class name, same argument
names, same attribute names — only `rngs=nnx.Rngs(params=0)` becomes
`key=jax.random.key(0)`, and `Linear` is handed to you the way `nnx.Linear`
is.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


class Linear:
    """Given to you, exactly as nnx.Linear is given to you in problem 15."""

    def __init__(self, d_in, d_out, *, key):
        self.kernel = jax.random.normal(key, (d_in, d_out)) / jnp.sqrt(d_in)
        self.bias = jnp.zeros((d_out,))

    def __call__(self, x):
        return x @ self.kernel + self.bias


class SwiGLUMLP:
    def __init__(self, d_model, d_ff, *, key):
        # Three independent keys: one reused would make gate and up identical,
        # collapsing SwiGLU into silu(z) * z.
        kg, ku, kd = jax.random.split(key, 3)
        self.gate_proj = Linear(d_model, d_ff, key=kg)
        self.up_proj = Linear(d_model, d_ff, key=ku)
        self.down_proj = Linear(d_ff, d_model, key=kd)

    def __call__(self, x):
        # The * is elementwise, so gate and up must share a width.
        return self.down_proj(jax.nn.silu(self.gate_proj(x)) * self.up_proj(x))

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

mlp = SwiGLUMLP(8, 16, key=jax.random.key(0))
print("gate", mlp.gate_proj.kernel.shape,
      " up", mlp.up_proj.kernel.shape,
      " down", mlp.down_proj.kernel.shape)

x = jax.random.normal(jax.random.key(1), (2, 5, 8))
print("out:", mlp(x).shape)

# The gate really does gate: zeroing it kills the output.
mlp.gate_proj.kernel = jnp.zeros_like(mlp.gate_proj.kernel)
mlp.gate_proj.bias = jnp.zeros_like(mlp.gate_proj.bias)
print("gate forced to 0 -> out is down(0):", jnp.allclose(mlp(x), mlp.down_proj(jnp.zeros((2, 5, 16)))))

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("mlp_pure")